# Notebook 05 – Edge Decoder Module

**Task 1.2 – Data Loader & Link Predictor Engineering**

This notebook defines and validates the **Edge Decoder** component used to score
candidate user–item pairs after the GNN encoder has produced node embeddings.

## What this notebook does
1. Defines `DotProductDecoder` – element-wise multiply then sum → scalar logit.
2. Defines `MLPDecoder` – concatenate embeddings → 2-layer MLP → scalar logit.
3. Validates both decoders on a synthetic mini-batch that mirrors real loader output.
4. Demonstrates end-to-end usage with `BCEWithLogitsLoss`.

### Design choice
Both decoders produce **raw (un-activated) logits** of shape `[batch_size]`.  
The loss function (`BCEWithLogitsLoss`) applies the sigmoid internally, which is
numerically more stable than computing `sigmoid → BCELoss`.

## 1. Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version : {torch.__version__}")

PyTorch version : 2.11.0+cpu


## 2. `DotProductDecoder`

**Formula:**
$$\hat{y}_i = \mathbf{u}_i \odot \mathbf{v}_i \cdot \mathbf{1} = \sum_d u_{i,d} \cdot v_{i,d}$$

This is equivalent to the dot product between each user–item embedding pair.
It has **zero learnable parameters** and relies entirely on the encoder to embed
users and items into a shared metric space where inner-product similarity is meaningful.

In [2]:
class DotProductDecoder(nn.Module):
    """Parameter-free edge decoder based on inner-product similarity.

    Given aligned user and item embedding matrices (one row per edge in the
    mini-batch), returns a 1-D tensor of raw logits.

    Args:
        user_emb (Tensor): shape [batch_size, emb_dim] – user embeddings.
        item_emb (Tensor): shape [batch_size, emb_dim] – item embeddings.
            Both tensors must have the same shape.

    Returns:
        logits (Tensor): shape [batch_size] – un-activated scores.
    """

    def forward(self, user_emb: torch.Tensor, item_emb: torch.Tensor) -> torch.Tensor:
        # Element-wise product then sum over the embedding dimension
        # (user_emb * item_emb) → [batch_size, emb_dim]
        # .sum(dim=-1)          → [batch_size]
        return (user_emb * item_emb).sum(dim=-1)

## 3. `MLPDecoder`

A shallow **2-layer MLP** decoder that concatenates the user and item embeddings
and maps them to a scalar logit:

$$\hat{y}_i = W_2 \, \text{ReLU}\!\left(W_1 [\mathbf{u}_i \| \mathbf{v}_i] + b_1\right) + b_2$$

The MLP adds learnable interaction weights, which can be helpful when the encoder
operates in a high-dimensional space and the simple dot product is insufficient.

In [3]:
class MLPDecoder(nn.Module):
    """Shallow MLP edge decoder.

    Concatenates the user and item embeddings and passes them through a
    2-layer fully-connected network to produce a scalar logit per edge.

    Architecture:
        Linear(2*emb_dim → hidden_dim) → ReLU → Dropout → Linear(hidden_dim → 1)

    Args:
        emb_dim   (int): Dimensionality of each node embedding.
        hidden_dim (int): Width of the hidden layer (default: 64).
        dropout   (float): Dropout probability applied after activation (default: 0.1).

    Returns:
        logits (Tensor): shape [batch_size] – un-activated scores.
    """

    def __init__(self, emb_dim: int, hidden_dim: int = 64, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * emb_dim, hidden_dim),   # concatenated input
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, 1),              # scalar output
        )

    def forward(self, user_emb: torch.Tensor, item_emb: torch.Tensor) -> torch.Tensor:
        # Concatenate along the feature dimension
        # [batch_size, emb_dim] + [batch_size, emb_dim] → [batch_size, 2*emb_dim]
        x = torch.cat([user_emb, item_emb], dim=-1)
        # MLP output: [batch_size, 1] → squeeze → [batch_size]
        return self.net(x).squeeze(-1)

## 4. Smoke Test – Synthetic Mini-Batch

We create a synthetic mini-batch that matches the shapes produced by
`LinkNeighborLoader` (from Notebook 04) to verify the decoders work correctly
before wiring them into the full encoder–decoder pipeline.

In [4]:
# ── Hyperparameters matching the real graph ───────────────────────────────────
BATCH_SIZE  = 256   # supervision edges per mini-batch
EMB_DIM     = 384   # must match encoder output dimension (= raw feature dim here)
HIDDEN_DIM  = 64    # MLP hidden width

# Synthetic embeddings: in practice these come from the GNN encoder
torch.manual_seed(42)
user_emb = torch.randn(BATCH_SIZE, EMB_DIM)  # [batch_size, 384]
item_emb = torch.randn(BATCH_SIZE, EMB_DIM)  # [batch_size, 384]

# Ground-truth labels: 50 % positive (1) / 50 % negative (0)
labels = torch.cat([
    torch.ones(BATCH_SIZE // 2),
    torch.zeros(BATCH_SIZE // 2),
])

print(f"user_emb shape : {user_emb.shape}")
print(f"item_emb shape : {item_emb.shape}")
print(f"labels   shape : {labels.shape}")

user_emb shape : torch.Size([256, 384])
item_emb shape : torch.Size([256, 384])
labels   shape : torch.Size([256])


In [5]:
# ── Test DotProductDecoder ────────────────────────────────────────────────────
dot_decoder = DotProductDecoder()

dot_logits = dot_decoder(user_emb, item_emb)

assert dot_logits.shape == (BATCH_SIZE,), \
    f"Expected shape ({BATCH_SIZE},) but got {dot_logits.shape}"
assert dot_logits.ndim == 1, "Output must be 1-D"

print("DotProductDecoder")
print(f"  Output shape : {dot_logits.shape}")
print(f"  dtype        : {dot_logits.dtype}")
print(f"  min / max    : {dot_logits.min():.4f} / {dot_logits.max():.4f}")
print(f"  Trainable parameters : {sum(p.numel() for p in dot_decoder.parameters())}")

DotProductDecoder
  Output shape : torch.Size([256])
  dtype        : torch.float32
  min / max    : -55.1910 / 39.7427
  Trainable parameters : 0


In [6]:
# ── Test MLPDecoder ───────────────────────────────────────────────────────────
mlp_decoder = MLPDecoder(emb_dim=EMB_DIM, hidden_dim=HIDDEN_DIM, dropout=0.1)

mlp_logits = mlp_decoder(user_emb, item_emb)

assert mlp_logits.shape == (BATCH_SIZE,), \
    f"Expected shape ({BATCH_SIZE},) but got {mlp_logits.shape}"
assert mlp_logits.ndim == 1, "Output must be 1-D"

n_params = sum(p.numel() for p in mlp_decoder.parameters())

print("MLPDecoder")
print(f"  Output shape           : {mlp_logits.shape}")
print(f"  dtype                  : {mlp_logits.dtype}")
print(f"  min / max              : {mlp_logits.min():.4f} / {mlp_logits.max():.4f}")
print(f"  Trainable parameters   : {n_params}")
print(f"  Architecture summary   :")
print(mlp_decoder)

MLPDecoder
  Output shape           : torch.Size([256])
  dtype                  : torch.float32
  min / max              : -0.5354 / 0.6412
  Trainable parameters   : 49281
  Architecture summary   :
MLPDecoder(
  (net): Sequential(
    (0): Linear(in_features=768, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


## 5. End-to-End Loss Computation

Demonstrate how the decoder logits plug into `BCEWithLogitsLoss` —
the standard loss for binary link prediction.

In [7]:
criterion = nn.BCEWithLogitsLoss()

# ── DotProduct path ───────────────────────────────────────────────────────────
dot_loss = criterion(dot_logits, labels)

# ── MLP path ──────────────────────────────────────────────────────────────────
mlp_loss = criterion(mlp_logits, labels)

print(f"DotProductDecoder BCE loss : {dot_loss.item():.4f}")
print(f"MLPDecoder        BCE loss : {mlp_loss.item():.4f}")

# Verify gradients flow through MLP decoder
mlp_loss.backward()
for name, p in mlp_decoder.named_parameters():
    assert p.grad is not None, f"No gradient for {name}"
print("\n✓ Gradients flow correctly through MLPDecoder.")

DotProductDecoder BCE loss : 8.5309
MLPDecoder        BCE loss : 0.7071

✓ Gradients flow correctly through MLPDecoder.


## 6. How the Decoder Fits into Training

Below is a **pseudocode sketch** of how the decoder is used inside the training loop
(implemented in Notebook 05 / 06):

```python
for batch in train_loader:
    # 1. GNN encoder → node embeddings
    node_emb = encoder(batch.x_dict, batch.edge_index_dict)

    # 2. Gather user / item embeddings for each supervision edge
    edge_idx = batch['user', 'reviews', 'item'].edge_label_index  # [2, batch_size]
    user_emb = node_emb['user'][edge_idx[0]]   # [batch_size, emb_dim]
    item_emb = node_emb['item'][edge_idx[1]]   # [batch_size, emb_dim]

    # 3. Decode → scalar logits
    logits = decoder(user_emb, item_emb)        # [batch_size]

    # 4. Compute loss and back-prop
    labels = batch['user', 'reviews', 'item'].edge_label.float()
    loss   = criterion(logits, labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
```

## 7. Summary

| Decoder | Parameters | Formula | Best used when |
|---|---|---|---|
| `DotProductDecoder` | 0 | $\sum_d u_d \cdot v_d$ | Encoder embeds into a good metric space |
| `MLPDecoder` | `2*emb_dim*hidden + hidden + hidden + 1` | MLP( [u ‖ v] ) | Extra learnable interaction weights needed |

Both decoders output **shape `[batch_size]` raw logits** — ready to be consumed
by `nn.BCEWithLogitsLoss` and the training loops in Notebooks 06 (GraphSAGE) and 07 (GCN).